# 06. Сводка результатов

Это основной reader-facing result notebook для курсовой. Он читает сохраненные public CSV/JSON files из `results/` и не запускает CLIP, Moment-DETR или heavy experiments.


## Main CLIP sweep на 1,000 queries

Основной финальный эксперимент - CLIP window/stride/aggregation sweep на фиксированной 1,000-query Charades-STA subset.


In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'results').exists():
    ROOT = ROOT.parent

def read_json(path):
    with open(ROOT / path, 'r', encoding='utf-8') as f:
        return json.load(f)

clip_1000 = pd.read_csv(ROOT / 'results/charades_sta_sweep_1000/summary.csv')
clip_1000[['config_name', 'window_size', 'stride', 'aggregation', 'evaluated_queries', 'R@1_IoU_0.3', 'R@1_IoU_0.5', 'R@1_IoU_0.7', 'mIoU', 'inference_time_sec']]


,config_name,window_size,stride,aggregation,evaluated_queries,R@1_IoU_0.3,R@1_IoU_0.5,R@1_IoU_0.7,mIoU,inference_time_sec
0,clip_w8_s4_mean,8.0,4.0,mean,1000,0.509,0.393,0.181,0.347777,309.686636
1,clip_w16_s8_mean,16.0,8.0,mean,1000,0.625,0.260,0.096,0.349690,22.592478
2,clip_w32_s16_mean,32.0,16.0,mean,1000,0.395,0.014,0.000,0.281429,21.506191
3,clip_w16_s8_max,16.0,8.0,max,1000,0.602,0.232,0.077,0.338619,20.922287


Вывод: `clip_w16_s8_mean` сильнее всего для coarse retrieval при IoU 0.3, а `clip_w8_s4_mean` сильнее всего для stricter localization при IoU 0.5 и 0.7. Эта таблица является центральным финальным результатом курсовой.


## Smoothing experiment на тех же 1,000 queries

Smoothing ablation использует ту же fixed subset и сохраненные CLIP embeddings. Он проверяет, улучшает ли moving-average smoothing frame-level CLIP scores качество localization.


In [2]:
smoothing = pd.read_csv(ROOT / 'results/charades_sta_smoothing_1000/summary.csv')
smoothing[['config_name', 'smoothing', 'evaluated_queries', 'R@1_IoU_0.3', 'R@1_IoU_0.5', 'R@1_IoU_0.7', 'mIoU']]


,config_name,smoothing,evaluated_queries,R@1_IoU_0.3,R@1_IoU_0.5,R@1_IoU_0.7,mIoU
0,clip_w8_s4_mean_none,none,1000,0.509,0.393,0.181,0.347777
1,clip_w8_s4_mean_moving_average_3,moving_average_3,1000,0.511,0.398,0.183,0.348520
2,clip_w8_s4_mean_moving_average_5,moving_average_5,1000,0.509,0.398,0.190,0.349421
3,clip_w16_s8_mean_none,none,1000,0.625,0.260,0.096,0.349690
4,clip_w16_s8_mean_moving_average_3,moving_average_3,1000,0.620,0.257,0.098,0.349082
5,clip_w16_s8_mean_moving_average_5,moving_average_5,1000,0.623,0.254,0.098,0.348085


Вывод: smoothing является дополнительной ablation. Он немного улучшает strict localization для short-window `w8/s4/mean`, но не является универсальным улучшением и не заменяет main CLIP sweep.


## CLIP vs Moment-DETR 50-query feasibility probe

Сравнение ниже использует фиксированную 50-query subset. Moment-DETR включен только как raw-video feasibility probe, а не как full official benchmark.


In [3]:
rows = []
for name in ['clip_w8_s4_mean', 'clip_w16_s8_mean', 'clip_w32_s16_mean', 'clip_w16_s8_max']:
    rows.append(read_json(f'results/clip_vs_moment_detr_50/{name}/run_config.json')['summary'])
rows.append({'model': 'Moment-DETR', 'config_name': 'raw_video_checkpoint', **read_json('results/moment_detr_charades_50/metrics.json')})
pd.DataFrame(rows)[['model', 'config_name', 'evaluated_queries', 'R@1_IoU_0.3', 'R@1_IoU_0.5', 'R@1_IoU_0.7', 'mIoU', 'inference_time_sec']]


,model,config_name,evaluated_queries,R@1_IoU_0.3,R@1_IoU_0.5,R@1_IoU_0.7,mIoU,inference_time_sec
0,CLIP,clip_w8_s4_mean,50,0.58,0.52,0.32,0.439378,14.223121
1,CLIP,clip_w16_s8_mean,50,0.68,0.36,0.08,0.400772,2.116203
2,CLIP,clip_w32_s16_mean,50,0.54,0.00,0.00,0.285429,2.091613
3,CLIP,clip_w16_s8_max,50,0.56,0.36,0.06,0.337459,2.093376
4,Moment-DETR,raw_video_checkpoint,50,0.44,0.32,0.04,0.263402,17.933695


Вывод: на этой небольшой 50-query subset CLIP дает сильные baseline numbers, а Moment-DETR успешно строит raw-video predictions. Comparison полезен для feasibility, но его нельзя читать как full Moment-DETR benchmark.


## Финальные выводы

Финальная public evidence line - Charades-STA. Основной результат - 1,000-query CLIP sweep; smoothing является controlled ablation на тех же данных; Moment-DETR сначала использовался как 50-query feasibility probe, а затем был оценен на полном Charades-STA test split: 3,720 queries / 1,334 videos. QVHighlights остается частью истории проекта и обсуждения ограничений, но не заявляется как completed benchmark.
